# 01 - Getting started with Delphos

This notebook is the shortest path from a trained Delphos checkpoint to candidate choice-model specifications.

You will learn how to:

- list datasets
- load the pretrained Delphos agent
- propose and estimate models
- inspect terms, parameters, and generated Apollo code
- save a proposal table for later review


## 1. Import Delphos


In [4]:
import delphos as dp

print("Delphos is ready")


Delphos is ready


## 2. See datasets

The package ships with example datasets that are already mapped to the trained Delphos catalogue.


In [ ]:
datasets = dp.list_datasets()
for item in datasets:
    print(f"{item.id:>2} | {item.name:<24} | {item.folder}")

# training and inference datasets (inference_datasets())
# Swissmetro wasnt used for training, so we will apply Delphos on Swissmtro


 1 | ApolloModeChoice         | dataset_1
 2 | SwissmetroRouteChoice    | dataset_2
 3 | Decisions                | dataset_3
 4 | Swissmetro               | dataset_4
 5 | NLModeChoice             | dataset_5
 6 | NorwayVTT                | dataset_6
 7 | Arentze2013              | dataset_7
 8 | SpainParkingchoice       | dataset_8
 9 | LondonModeChoice         | dataset_9
10 | Optima                   | dataset_10
11 | VanCranenburghVOT        | dataset_11


## 3. Load unseen dataset and the trained agent

The default checkpoint is the production multitask checkpoint included in `checkpoints/full_agent_task_10_seed_123`.


In [ ]:

dataset = dp.load_dataset("Swissmetro")

agent = dp.load_agent()

print(dataset)
print(agent.agent.summary())


Task(name='Swissmetro', alternatives=3, attributes=5, covariates=11)
{'agent': 'DelphosAgent', 'mode': 'inference', 'encoder_kind': 'deepset', 'state_dim': 64, 'num_actions': 297, 'z_cfg': {'K': 7, 'T': 3, 'G': 2, 'C': 7, 'd_att': 16, 'd_tr': 8, 'd_taste': 8, 'd_cov': 16, 'd_term': 64, 'd_state': 128, 'context_dim': 0, 'head_flag': False, 'pooling': 'mean', 'attention_heads': 4, 'attention_layers': 1, 'attention_dropout': 0.0}, 'device': 'cpu'}


## 4. Propose models without estimation

`estimate=False` is the default. This is fast because Delphos only searches the modelling space and builds Apollo-ready specifications. It does not call R.


In [6]:
models = agent.propose(
    dataset,
    n_models=5,
    estimate=True
)

models.to_dataframe()
# do not show the 


Apollo ignition sequence completed
Several observations per individual detected based on the value of id.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Apollo ignition sequence completed
Several observations per individual detected based on the value of id.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Apollo ignition sequence completed
Several observations per individual detected based on the value of id.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Apollo ignition sequence completed
Several observations per individual detected based on the value of id.
  Setting panelData in apollo_control set to TRUE.
All checks on apollo_control completed.
All checks on database completed.
Apollo ignition sequence completed
Several observations per individual detec

,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices,...,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,4,Swissmetro,1110_2124_3212_4110_5000_6110_7000,10,topk,0,True,0.080552,5,"[125, 17, 215, 27, 74, 75, 106, 105, 201, 21]",...,0.266062,0.264052,0.128452,0.126406,10251.258636,10346.738089,-7.165797,1.132897,14,0
1,4,Swissmetro,1110_2212_3212_4324_5000_6110_7000,10,topk,1,True,0.096403,5,"[215, 74, 21, 106, 201, 27, 75, 215, 149, 201]",...,0.278470,0.276173,0.143186,0.140799,10082.426811,10191.546185,-1.686022,1.417425,16,0
2,4,Swissmetro,1110_2124_3210_4214_5000_6110_7000,10,topk,2,True,0.077291,5,"[21, 125, 17, 215, 27, 201, 106, 73, 21, 125]",...,0.263514,0.261791,0.125426,0.123721,10282.754558,10364.594089,-9.808499,0.697092,12,0
3,4,Swissmetro,1110_2121_3212_4111_5000_6110_7000,10,topk,3,True,0.082602,5,"[21, 125, 106, 17, 215, 21, 27, 18, 201, 75]",...,0.267665,0.265511,0.130355,0.128138,10230.934350,10333.233764,-7.155435,2.857019,15,0
4,4,Swissmetro,1110_2212_3210_4222_5000_6126_7000,10,topk,4,True,0.097835,5,"[73, 27, 17, 21, 215, 106, 17, 27, 149, 131]",...,0.279593,0.277152,0.144519,0.141962,10068.785960,10184.725296,-1.118034,1.279611,17,0


## 5. Inspect the first proposal

Motivation -> specify the goal. LL and BIC


In [7]:
proposal = models.proposals[0]

print("Specification key:", proposal.specification_key)
print("Episode length:", proposal.episode_length)
print("Action indices:", proposal.action_indices)
print("Terms:")
for term in proposal.terms:
    print(term)


Specification key: 1110_2124_3212_4110_5000_6110_7000
Episode length: 10
Action indices: [125, 17, 215, 27, 74, 75, 106, 105, 201, 21]
Terms:
Term(attribute_id=1, transform_id=1, taste_id=1, covariate_id=0)
Term(attribute_id=2, transform_id=1, taste_id=2, covariate_id=4)
Term(attribute_id=3, transform_id=2, taste_id=1, covariate_id=2)
Term(attribute_id=4, transform_id=1, taste_id=1, covariate_id=0)
Term(attribute_id=6, transform_id=1, taste_id=1, covariate_id=0)


## 6. Inspect generated Apollo components

These objects are what Delphos sends to the environment when `estimate=True`.


In [8]:
apollo_spec = proposal.apollo_specification

print("Number of parameters:", apollo_spec.n_parameters)
print("First parameters:", apollo_spec.parameter_names[:10])
print()
print("Utility code preview:")
print()
print(apollo_spec.utility_code)


Number of parameters: 16
First parameters: ['ASC_TRAIN', 'ASC_SM', 'ASC_CAR', 'b_TRAIN_time', 'b_SM_time', 'b_CAR_time', 'b_cost_generic_log_income_1', 'b_cost_generic_log_income_2', 'b_cost_generic_log_income_3', 'b_cost_generic_log_income_4']

Utility code preview:

V <- list()

V[["TRAIN"]] <-
      ASC_TRAIN +
      b_TRAIN_time * train_tt_scaled +
      b_cost_generic_log_income_1 * (income == 1) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_2 * (income == 2) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_3 * (income == 3) * log(1+train_cost_scaled) +
      b_cost_generic_log_income_4 * (income == 4) * log(1+train_cost_scaled) +
      b_TRAIN_headway_box_cox_purpose_1 * (purpose == 1) * ((train_he_scaled^L_headway - 1) / L_headway) +
      b_TRAIN_headway_box_cox_purpose_2 * (purpose == 2) * ((train_he_scaled^L_headway - 1) / L_headway)

V[["SM"]] <-
      ASC_SM +
      b_SM_time * sm_tt_scaled +
      b_cost_generic_log_income_1 * (income == 1) * log

## 7. Save proposals

This is a useful pattern when you want to inspect models first, then estimate them later.


In [8]:
output_path = "getting_started_proposals.csv"
models.to_dataframe().to_csv(output_path, index=False)
print(f"Saved {len(models)} proposals to {output_path}")


Saved 5 proposals to getting_started_proposals.csv
